# Оценка Retrieval и Reranking

Сравнивает качество поиска ДО и ПОСЛЕ реранкинга на золотом датасете.

**Метрики:**
- Recall@1, @3, @5 — доля вопросов с релевантным чанком в топ-K
- MRR — средняя позиция первого релевантного документа
- По типам вопросов и сложности (single-hop vs multi-hop)

In [ ]:
# Конфигурация
PROJECT_ROOT = "c:/Users/kozlovaalek/Projects/LLM"

# API URLs
OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3:latest"
QDRANT_URL = "http://localhost:6333"
COLLECTION = "CEN1_ТИ_3-48200234-05.1-12-2020_Очистка_электролита_от_примесей"
RERANKER_URL = "http://localhost:8082"

# Параметры
# Для честного сравнения retrieval vs reranking берем одинаковый top_k
RETRIEVE_TOP_K = 20
RERANK_TOP_K = 20  # ← теперь возвращаем все 20, чтобы увидеть только переупорядочивание

GOLDEN_DATASET_PATH = f"{PROJECT_ROOT}/data/04_golden_dataset/qa_pairs_v2_podpunkti_golden.jsonl"

print("✓ Конфигурация готова")

In [2]:
import os
import json
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt
from datetime import datetime
from urllib.parse import quote

print("✓ Импорты готовы")

✓ Импорты готовы


In [3]:
# Загрузка золотого датасета
qa_pairs = []
with open(GOLDEN_DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            qa_pairs.append(json.loads(line))

print(f"✓ Загружено {len(qa_pairs)} QA-пар")
print(f"\n📊 Статистика:")
types = Counter(qa["type"] for qa in qa_pairs)
multi_hop_count = sum(1 for qa in qa_pairs if qa.get("multi_hop_required", False))
print(f"  Single-hop вопросов: {len(qa_pairs) - multi_hop_count}")
print(f"  Multi-hop вопросов:  {multi_hop_count}")

✓ Загружено 85 QA-пар

📊 Статистика:
  Single-hop вопросов: 54
  Multi-hop вопросов:  31


In [ ]:
COLLECTION_ENCODED = quote(COLLECTION, safe='')

def embed_query(text):
    r = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": EMBED_MODEL, "input": text},
        timeout=120
    )
    r.raise_for_status()
    return r.json()["embeddings"][0]

def search_qdrant(vector, top_k):
    r = requests.post(
        f"{QDRANT_URL}/collections/{COLLECTION_ENCODED}/points/search",
        json={"vector": vector, "limit": top_k, "with_payload": True},
        timeout=30
    )
    r.raise_for_status()
    return r.json()["result"]

def rerank_results(question, hits, top_k):
    docs = [h.get("payload", {}).get("text", "")[:512] for h in hits]
    r = requests.post(
        f"{RERANKER_URL}/rerank",
        json={"query": question, "docs": docs, "top_k": top_k},
        timeout=60
    )
    r.raise_for_status()
    results = r.json().get("results", [])
    reranked = []
    for item in results:
        idx = item.get("index", -1)
        if 0 <= idx < len(hits):
            reranked.append({
                "chunk_id": hits[idx].get("payload", {}).get("chunk_id"),
                "score": item.get("score"),
            })
    return reranked

# Проверка подключения
print(f"OLLAMA_URL:    {OLLAMA_URL}")
print(f"QDRANT_URL:    {QDRANT_URL}")
print(f"RERANKER_URL:  {RERANKER_URL}")
print(f"COLLECTION:    {COLLECTION}")
print()

try:
    emb = embed_query("тест")
    print(f"✓ Ollama OK (dim={len(emb)})")
except Exception as e:
    print(f"✗ Ollama: {e}")

try:
    r = requests.get(f"{QDRANT_URL}/collections/{COLLECTION_ENCODED}", timeout=5)
    points = r.json()["result"]["points_count"]
    print(f"✓ Qdrant OK ({points} точек)")
except Exception as e:
    print(f"✗ Qdrant: {e}")

try:
    r = requests.post(
        f"{RERANKER_URL}/rerank",
        json={"query": "тест", "docs": ["тест текст"], "top_k": 1},
        timeout=10
    )
    print(f"✓ Reranker OK")
except Exception as e:
    print(f"✗ Reranker: {e}")

In [ ]:
def recall_at_k(results, k):
    """Recall@K"""
    found = sum(1 for r in results if r.get("rank") is not None and r["rank"] <= k)
    return found / len(results) if results else 0

def mrr(results):
    """Mean Reciprocal Rank"""
    rr_list = [1 / r["rank"] if r.get("rank") is not None else 0 for r in results]
    return np.mean(rr_list) if rr_list else 0

def ndcg_at_k(results_list, k):
    """NDCG@K — учитывает позиции всех релевантных чанков"""
    def dcg(ranks):
        return sum(1.0 / np.log2(r + 1) for r in ranks if r is not None and r <= k)

    def ideal_dcg(num_relevant):
        return sum(1.0 / np.log2(i + 1) for i in range(1, min(num_relevant, k) + 1))

    ndcg_scores = []
    for result in results_list:
        ranks = result.get("all_ranks", [result.get("rank")] if result.get("rank") else [])
        ideal = ideal_dcg(len(ranks))
        actual = dcg(ranks)
        ndcg_scores.append(actual / ideal if ideal > 0 else 0)

    return np.mean(ndcg_scores) if ndcg_scores else 0

def find_ranks(chunk_ids, reference_chunks):
    """Найти позиции всех релевантных чанков"""
    ranks = []
    for rank, chunk_id in enumerate(chunk_ids, 1):
        if chunk_id in reference_chunks:
            ranks.append(rank)
    return ranks if ranks else None

print("✓ Метрики инициализированы")

In [ ]:
# Оценка retrieval и reranking
results_before = []
results_after = []

for idx, qa in enumerate(qa_pairs, 1):
    question = qa["question"]
    reference_chunks = set(qa.get("source_chunks", []))
    
    try:
        # Embedding
        emb = embed_query(question)
        
        # Retrieval — берем chunk_id из payload
        hits = search_qdrant(emb, RETRIEVE_TOP_K)
        chunk_ids_before = [h.get("payload", {}).get("chunk_id") for h in hits]
        ranks_before = find_ranks(chunk_ids_before, reference_chunks)
        rank_before = ranks_before[0] if ranks_before else None
        
        results_before.append({
            "qa_id": qa["id"],
            "type": qa.get("type", "unknown"),
            "multi_hop": qa.get("multi_hop_required", False),
            "rank": rank_before,
            "all_ranks": ranks_before or [],
        })
        
        # Reranking
        reranked = rerank_results(question, hits, RERANK_TOP_K)
        if not reranked:
            reranked = [{"chunk_id": cid} for cid in chunk_ids_before[:RERANK_TOP_K]]
        
        chunk_ids_after = [r.get("chunk_id") for r in reranked]
        ranks_after = find_ranks(chunk_ids_after, reference_chunks)
        rank_after = ranks_after[0] if ranks_after else None
        
        results_after.append({
            "qa_id": qa["id"],
            "type": qa.get("type", "unknown"),
            "multi_hop": qa.get("multi_hop_required", False),
            "rank": rank_after,
            "all_ranks": ranks_after or [],
        })
        
        # Промежуточный вывод
        rank_b_str = str(rank_before) if rank_before is not None else "✗"
        rank_a_str = str(rank_after) if rank_after is not None else "✗"
        
        # Индикатор изменения
        if rank_before == rank_after:
            change = "="
        elif rank_before is None and rank_after is not None:
            change = "↑↑"
        elif rank_before is not None and rank_after is None:
            change = "↓↓"
        elif rank_after < rank_before:
            change = "↑"
        else:
            change = "↓"
        
        ref_str = str(sorted(reference_chunks)) if reference_chunks else "[]"
        print(f"[{idx:3d}/{len(qa_pairs)}] {qa['id']:15} {qa.get('type', 'unknown'):25} | ref={ref_str:12} | rank: {rank_b_str:>3} → {rank_a_str:>3} {change}")
        
    except Exception as e:
        print(f"[{idx:3d}/{len(qa_pairs)}] Ошибка {qa['id']}: {str(e)[:80]}")

print(f"\n✓ Оценено {len(results_before)} вопросов")

In [ ]:
# Итоговое сравнение
display(Markdown("## 📊 Сравнение: До реранкинга vs После"))

metrics_comparison = []
for k in [1, 3, 5, 10, 20]:
    recall_b = recall_at_k(results_before, k)
    recall_a = recall_at_k(results_after, k)
    improvement = (recall_a - recall_b) * 100
    
    metrics_comparison.append({
        "Метрика": f"Recall@{k}",
        "До реранкинга": f"{recall_b:.4f}",
        "После реранкинга": f"{recall_a:.4f}",
        "Улучшение": f"{improvement:+.1f}%",
    })

mrr_b = mrr(results_before)
mrr_a = mrr(results_after)
improvement = (mrr_a - mrr_b) * 100

metrics_comparison.append({
    "Метрика": "MRR",
    "До реранкинга": f"{mrr_b:.4f}",
    "После реранкинга": f"{mrr_a:.4f}",
    "Улучшение": f"{improvement:+.1f}%",
})

for k in [3, 5, 10, 20]:
    ndcg_b = ndcg_at_k(results_before, k)
    ndcg_a = ndcg_at_k(results_after, k)
    improvement = (ndcg_a - ndcg_b) * 100
    
    metrics_comparison.append({
        "Метрика": f"NDCG@{k}",
        "До реранкинга": f"{ndcg_b:.4f}",
        "После реранкинга": f"{ndcg_a:.4f}",
        "Улучшение": f"{improvement:+.1f}%",
    })

df_metrics = pd.DataFrame(metrics_comparison)
display(df_metrics.set_index("Метрика"))

In [ ]:
# По типам вопросов
display(Markdown("## 🎯 По типам вопросов (Recall@1)"))

by_type_before = defaultdict(list)
by_type_after = defaultdict(list)

for b, a in zip(results_before, results_after):
    by_type_before[b["type"]].append(b)
    by_type_after[a["type"]].append(a)

by_type_data = []
for qtype in sorted(by_type_before.keys()):
    r1_b = recall_at_k(by_type_before[qtype], 1)
    r1_a = recall_at_k(by_type_after[qtype], 1)
    improvement = (r1_a - r1_b) * 100
    count = len(by_type_before[qtype])
    
    by_type_data.append({
        "Тип": qtype,
        "Вопросов": count,
        "Recall@1 (до)": f"{r1_b:.3f}",
        "Recall@1 (после)": f"{r1_a:.3f}",
        "Улучшение": f"{improvement:+.1f}%",
    })

df_by_type = pd.DataFrame(by_type_data)
display(df_by_type.set_index("Тип"))

In [ ]:
# Multi-hop анализ
display(Markdown("## 🔗 Single-hop vs Multi-hop"))

single_hop_before = [r for r in results_before if not r["multi_hop"]]
multi_hop_before = [r for r in results_before if r["multi_hop"]]

single_hop_after = [r for r in results_after if not r["multi_hop"]]
multi_hop_after = [r for r in results_after if r["multi_hop"]]

multi_hop_data = []

for name, before, after in [("Single-hop", single_hop_before, single_hop_after),
                             ("Multi-hop", multi_hop_before, multi_hop_after)]:
    if before:
        r1_b = recall_at_k(before, 1)
        r1_a = recall_at_k(after, 1)
        improvement = (r1_a - r1_b) * 100
        
        multi_hop_data.append({
            "Тип": name,
            "Вопросов": len(before),
            "Recall@1 (до)": f"{r1_b:.3f}",
            "Recall@1 (после)": f"{r1_a:.3f}",
            "Улучшение": f"{improvement:+.1f}%",
        })

df_multi_hop = pd.DataFrame(multi_hop_data)
display(df_multi_hop.set_index("Тип"))

In [ ]:
# Примеры, где реранкер помог
display(Markdown("## ✅ Примеры улучшений от реранкера"))

improved = []
for b, a in zip(results_before, results_after):
    if b["rank"] is None and a["rank"] is not None:
        # Не было найдено, потом нашли
        improved.append((b, a, "notfound_to_found"))
    elif b["rank"] is not None and a["rank"] is not None and a["rank"] < b["rank"]:
        # Улучшилась позиция
        improved.append((b, a, "rank_improved"))

if improved:
    print(f"Всего улучшений: {len(improved)}\n")
    for b, a, reason in improved[:5]:
        qa = next(q for q in qa_pairs if q["id"] == b["qa_id"])
        print(f"📌 {b['qa_id']} ({b['type']})")
        print(f"Q: {qa['question'][:80]}...")
        if reason == "notfound_to_found":
            print(f"Результат: не найдено → ранг {a['rank']}")
        else:
            print(f"Результат: ранг {b['rank']} → {a['rank']}")
        print()
else:
    print("Нет улучшений от реранкера")

In [ ]:
# Примеры, где реранкер ухудшил
display(Markdown("## ❌ Примеры где реранкер ухудшил результаты"))

degraded = []
for b, a in zip(results_before, results_after):
    if b["rank"] is not None and a["rank"] is None:
        # Было найдено, потом потеряли
        degraded.append((b, a, "lost"))
    elif b["rank"] is not None and a["rank"] is not None and a["rank"] > b["rank"]:
        # Ухудшилась позиция
        degraded.append((b, a, "rank_worse"))

if degraded:
    print(f"Всего ухудшений: {len(degraded)}\n")
    for b, a, reason in degraded[:5]:
        qa = next(q for q in qa_pairs if q["id"] == b["qa_id"])
        print(f"📌 {b['qa_id']} ({b['type']})")
        print(f"Q: {qa['question'][:80]}...")
        if reason == "lost":
            print(f"Результат: ранг {b['rank']} → потеряно")
        else:
            print(f"Результат: ранг {b['rank']} → {a['rank']}")
        print()
else:
    print("Никаких ухудшений от реранкера")

In [ ]:
# Визуализация
display(Markdown("## 📈 Распределение позиций"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# До реранкинга
ranks_before = [r["rank"] for r in results_before if r["rank"] is not None]
axes[0].hist(ranks_before, bins=range(1, max(ranks_before or [1])+2), edgecolor='black')
axes[0].set_title(f"До реранкинга (найдено: {len(ranks_before)}/{len(results_before)})")
axes[0].set_xlabel("Позиция первого релевантного чанка")
axes[0].set_ylabel("Количество вопросов")
axes[0].grid(True, alpha=0.3)

# После реранкинга
ranks_after = [r["rank"] for r in results_after if r["rank"] is not None]
axes[1].hist(ranks_after, bins=range(1, max(ranks_after or [1])+2), edgecolor='black', color='green')
axes[1].set_title(f"После реранкинга (найдено: {len(ranks_after)}/{len(results_after)})")
axes[1].set_xlabel("Позиция первого релевантного чанка")
axes[1].set_ylabel("Количество вопросов")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()